# Week 4 — power and sample size

**27200 Data-driven Bioengineering · group assignment 3**

Everything the assignment needs, in one notebook. **Run the setup cell, then jump to the section for
your case** — the sections are independent and each one is a worked example you can retarget by
changing the numbers at the top.

| Section | Use it for | Case |
|---|---|---|
| 1 | Two groups, one measurement each | 1 |
| 2 | Repeated measurements of the same unit — which n? | 2, 6 |
| 3 | Two proportions (yes/no outcomes) | 3 |
| 4 | Thousands of tests at once | 4 |
| 5 | Paired designs — everyone measured under every condition | 5 |
| 6 | More than two groups, and what to do when the data are not normal | 6 |
| 7 | **Simulation** — when there is no formula | any |

> **The four quantities.**
> Power calculations connect four things: the **effect size** (how big a difference, relative to the
> noise), the **sample size**, the **significance level** α, and the **power** (the chance of
> detecting a real effect). **Fix any three and the fourth is determined.** Every function below does
> exactly this — leave out the one you want and it solves for it.
>
> By convention: α = 0.05, power = 0.80. Power 0.80 means that if the effect is really there at the
> size you assumed, you will miss it one time in five.

---
## 0 · Setup

Run this once. `power.t.test()`, `power.prop.test()` and `power.anova.test()` are part of base R;
`pwr` adds unequal group sizes and proportions expressed as effect sizes.

In [ ]:
install.packages("pwr")       # takes about a minute
library(pwr)
set.seed(27200)               # reproducible random numbers
cat("ready\n")

---
## 1 · Two groups, one measurement each

The standard two-sample t-test design. Give `power.t.test()` any three of `n`, `delta`, `sd`,
`power` and it solves for the missing one — **leave that one out entirely**, do not set it to NULL.

The numbers below are Case 1's; replace them with your own.

In [ ]:
delta = 10      # smallest difference that matters, in the units you measure
sd    = 18      # standard deviation of the measurement, from pilot data or the literature

# n left out -> solves for n (per group)
power.t.test(delta = delta, sd = sd, sig.level = 0.05, power = 0.80,
             type = "two.sample")

### Leaving out a different quantity

Same function, different blank. This answers "what can I detect with the patients I actually have?"
— the question your constraint will force on you.

In [ ]:
n_available = 40

# power left out -> what power do I have with the n I can get?
p = power.t.test(n = n_available, delta = delta, sd = sd, sig.level = 0.05,
                 type = "two.sample")
cat("power with n =", n_available, "per group:", round(p$power, 2), "\n")

# delta left out -> what is the smallest difference I could detect?
dd = power.t.test(n = n_available, sd = sd, sig.level = 0.05, power = 0.80,
                  type = "two.sample")
cat("smallest detectable difference:", round(dd$delta, 1), "units\n")

### Dropout

Sample size gives the number who must **finish**. Inflate for the ones who will not.

In [ ]:
n_complete = ceiling(power.t.test(delta = delta, sd = sd, sig.level = 0.05,
                                  power = 0.80, type = "two.sample")$n)
dropout    = 0.15
n_enrol    = ceiling(n_complete / (1 - dropout))

cat("must complete:", n_complete, "per group\n")
cat("must enrol:   ", n_enrol,    "per group (", 2*n_enrol, "total )\n")

### The power curve

The single most useful figure for this assignment: power against sample size, with your constraint
drawn on it.

In [ ]:
ns = 5:120
pw      = sapply(ns, function(n) power.t.test(n=n, delta=delta,   sd=sd, sig.level=0.05)$power)
pw_half = sapply(ns, function(n) power.t.test(n=n, delta=delta/2, sd=sd, sig.level=0.05)$power)

plot(ns, pw, type="l", lwd=2, ylim=c(0,1), xlab="n per group", ylab="power")
lines(ns, pw_half, lwd=2, col="grey50")
abline(h=0.80, lty=2)                       # the 80% convention
abline(v=n_available, lty=3, col="red", lwd=2)   # what you can actually get
legend("bottomright", c("assumed effect","half that effect"),
       col=c("black","grey50"), lwd=2, bty="n")

---
## 2 · Repeated measurements of the same unit

**This is the one that catches people.** If you run three bioreactors and take ten samples from
each, you have thirty measurements but you do **not** have n = 30. You have n = 3. The ten samples
tell you how precisely you measured *that reactor*, not how much reactors differ from each other —
and the second question is the one your experiment is asking.

The rule: **n is the number of independent units you could have replaced with a different one.**
Different run, different animal, different patient, different overnight culture. Repeated
measurements of one unit are *technical replicates*, and they belong inside that unit's mean.

In [ ]:
sd_between = 0.35    # SD between independent units (different runs of the same process)
sd_within  = 0.12    # SD between repeated measurements of ONE unit (assay noise)
delta      = 0.5     # smallest difference worth detecting

for (k in c(1, 2, 5, 10, 50)) {
  se = sqrt(sd_between^2 + sd_within^2 / k)     # SE of one unit's mean
  n  = power.t.test(delta = delta, sd = se, sig.level = 0.05, power = 0.80)$n
  cat(sprintf("%3d measurements per unit -> effective SD %.3f -> %3.0f UNITS per group\n",
              k, se, ceiling(n)))
}

n_wrong = power.t.test(delta = delta, sd = sd_within, sig.level = 0.05, power = 0.80)$n
cat("\nIf you wrongly treat each measurement as independent:", ceiling(n_wrong), "'samples' per group.\n")
cat("That is the error. It makes an underpowered experiment look comfortably powered.\n")

Notice how flat that list is. Once unit-to-unit variation dominates, extra measurements of the same
unit buy almost nothing — the money belongs in **more units**. Quote that comparison in your report:
it is a design decision with a number attached.

---
## 3 · Two proportions

When the outcome is yes/no there is no standard deviation — the rate sets the variability.
`power.prop.test()` takes the two proportions directly.

In [ ]:
power.prop.test(p1 = 0.02, p2 = 0.03, sig.level = 0.05, power = 0.80)

# Rare outcomes are expensive: the same relative increase costs far more at a low baseline rate.
for (pp in list(c(0.02,0.03), c(0.02,0.04), c(0.20,0.30))) {
  n = power.prop.test(p1 = pp[1], p2 = pp[2], sig.level = 0.05, power = 0.80)$n
  cat(sprintf("  %4.0f%% -> %4.0f%%: %7.0f per group\n", 100*pp[1], 100*pp[2], ceiling(n)))
}

### Unequal group sizes

Registry and cohort studies rarely have equal groups. `pwr.2p2n.test()` takes the two group sizes
separately. Note how the benefit saturates: past about 4 or 5 controls per case, extra controls buy
almost nothing — the smaller group is what limits you.

In [ ]:
h = ES.h(0.03, 0.02)          # effect size for two proportions (Cohen's h)
cat("Cohen's h =", round(h, 4), "\n\n")

grid = seq(100, 6000, by = 10)          # candidate sizes for the exposed group
for (ratio in c(1, 2, 4, 8, 32)) {
  pw = sapply(grid, function(n) pwr.2p2n.test(h = h, n1 = n, n2 = ratio*n,
                                              sig.level = 0.05)$power)
  n1 = grid[which(pw >= 0.80)[1]]       # the first size that reaches 80% power
  cat(sprintf("  %2d unexposed per exposed: %7.0f exposed needed\n", ratio, n1))
}

cat("\npower with 1200 exposed at 32:1 ->",
    round(pwr.2p2n.test(h = h, n1 = 1200, n2 = 32*1200, sig.level = 0.05)$power, 2), "\n")

---
## 4 · Thousands of tests at once

Test 20,000 genes at α = 0.05 and roughly 1,000 come out significant with nothing going on.
Correcting makes the threshold much stricter — and a stricter threshold needs a **bigger sample** to
clear. That is the cost of a genome-scale screen, and usually the reason such screens are
underpowered.

In [ ]:
n_tests = 20000
delta   = 1.0      # log2 fold change we want to detect
sd      = 0.5      # replicate SD on the log2 scale

for (a in c(0.05, 0.05/n_tests)) {
  n = power.t.test(delta = delta, sd = sd, sig.level = a, power = 0.80)$n
  cat(sprintf("alpha = %.2e -> n = %3.0f per group\n", a, ceiling(n)))
}

cat("\n")
for (n in c(4, 6, 10, 21)) {
  p = power.t.test(n = n, delta = delta, sd = sd, sig.level = 0.05/n_tests)$power
  cat(sprintf("  power at n = %2d per group (Bonferroni): %.3f\n", n, p))
}

Bonferroni is the strict end. Benjamini–Hochberg (FDR) is what genomics actually uses, and it has
no simple formula — so simulate it. The function below builds a whole experiment: 20,000 genes, a
few hundred genuinely changed, and counts how many survive FDR correction.

In [ ]:
fdr_power = function(n_per_group, n_tests = 20000, n_true = 200,
                     log2fc = 1.0, sd = 0.5, reps = 5) {
  # takes about a minute
  found = c(); false = c()
  for (r in 1:reps) {
    a = matrix(rnorm(n_tests*n_per_group, 0, sd), nrow = n_tests)
    b = matrix(rnorm(n_tests*n_per_group, 0, sd), nrow = n_tests)
    b[1:n_true, ] = b[1:n_true, ] + log2fc          # the genes that really changed
    # a vectorised two-sample t-test: one t value per row, all 20,000 at once
    se = sqrt((apply(a, 1, var) + apply(b, 1, var)) / n_per_group)
    t  = (rowMeans(b) - rowMeans(a)) / se
    p  = 2 * pt(-abs(t), df = 2*n_per_group - 2)
    sig = p.adjust(p, method = "BH") < 0.05
    found = c(found, sum(sig[1:n_true]) / n_true)
    false = c(false, sum(sig[(n_true+1):n_tests]))
  }
  c(mean(found), mean(false))
}

for (n in c(3, 6, 10)) {
  res = fdr_power(n)
  cat(sprintf("n = %2d per group: %5.1f%% of real changes found, %5.1f false positives\n",
              n, 100*res[1], res[2]))
}

---
## 5 · Paired designs

If the same subject is measured under both conditions, each subject is their own control and the
comparison is made *within* subjects. The standard deviation that matters is the **within-subject**
one, which is nearly always far smaller. That is why crossover designs need so few people.

Use `type = "paired"`, and remember `n` is then the number of **subjects**, not measurements.

In [ ]:
delta      = 40     # smallest meaningful difference between two conditions
sd_within  = 45     # same person, same condition, different day
sd_between = 90     # different people

n_paired = power.t.test(delta = delta, sd = sd_within, sig.level = 0.05,
                        power = 0.80, type = "paired")$n
sd_total = sqrt(sd_between^2 + sd_within^2)
n_parallel = power.t.test(delta = delta, sd = sd_total, sig.level = 0.05,
                          power = 0.80, type = "two.sample")$n

cat("paired   (everyone does both):", ceiling(n_paired), "subjects in total\n")
cat("parallel (two separate groups):", ceiling(n_parallel), "per group,",
    2*ceiling(n_parallel), "in total\n")
cat("\npairing saves a factor of",
    round(2*ceiling(n_parallel)/ceiling(n_paired)), "in people.\n")

### Several conditions means several comparisons

Five conditions give 10 pairwise comparisons, so the threshold tightens and n rises. Decide *in
advance* whether you need every pair or only each condition against a reference — that choice alone
changes how many people you must recruit.

In [ ]:
k = 5
n_all_pairs  = choose(k, 2)      # every condition against every other
n_vs_control = k - 1             # every condition against one reference

for (m in c(1, n_all_pairs, n_vs_control)) {
  n = power.t.test(delta = delta, sd = sd_within, sig.level = 0.05/m,
                   power = 0.80, type = "paired")$n
  cat(sprintf("%2d comparison(s): alpha = %.4f -> %3.0f subjects\n", m, 0.05/m, ceiling(n)))
}

---
## 6 · More than two groups, and non-normal data

For an overall "do these k groups differ at all?" test, `power.anova.test()` wants the variance
*between* group means and the variance *within* groups. It returns n **per group**.

In [ ]:
k = 5
group_means = c(0, 1, 2, 3, 4)      # expected mean per dose (log10 CFU below control)
sd_within   = 0.9                   # SD between independent biological repeats

power.anova.test(groups = k,
                 between.var = var(group_means),
                 within.var  = sd_within^2,
                 sig.level = 0.05, power = 0.80)

**But the omnibus test is rarely the question you care about.** "Do these five doses differ at
all?" is easy to answer. "Which is the lowest dose that still works?" is a set of pairwise
comparisons between neighbouring doses — a much smaller difference, tested several times, so
corrected — and *that* is what should set your sample size.

Design for the comparison you actually want to make, not for the omnibus test.

In [ ]:
delta_adjacent = 1.0          # difference between two neighbouring doses
m = k - 1                     # comparisons against the control

n_one  = power.t.test(delta = delta_adjacent, sd = sd_within, sig.level = 0.05,   power = 0.80)$n
n_corr = power.t.test(delta = delta_adjacent, sd = sd_within, sig.level = 0.05/m, power = 0.80)$n

cat(sprintf("one pairwise comparison, uncorrected: %3.0f per group\n", ceiling(n_one)))
cat(sprintf("%d comparisons, Bonferroni:            %3.0f per group\n", m, ceiling(n_corr)))

### When the data are not normal

An ANOVA or t-test assumes roughly normal residuals and similar spread per group. **Check before you
trust them**, and write down in advance what you will do if the check fails.

Three outcomes, three responses:

- **Skewed but continuous** (counts, concentrations, CFU/mL) — take logs first. A log-normal
  quantity becomes normal, and a "fold change" becomes a difference, which is what the test compares.
- **Still not normal, or ordinal** — use a rank-based test: `wilcox.test()` instead of `t.test()`,
  `kruskal.test()` instead of ANOVA, with Dunn's test for the follow-up pairs. Ranks do not care
  about shape.
- **Censored values** — measurements reported as "below the detection limit" are not numbers, and no
  transformation makes them into numbers. Rank-based tests cope (ties at the bottom); means do not.

In [ ]:
x = rlnorm(60, meanlog = 3, sdlog = 1.0)    # skewed, the way a CFU/mL count behaves

par(mfrow = c(1, 3))
hist(x, breaks = 15, main = "raw: skewed", xlab = "")
qqnorm(x, main = "Q-Q, raw"); qqline(x)
qqnorm(log10(x), main = "Q-Q, after log10"); qqline(log10(x))
par(mfrow = c(1, 1))

# Shapiro-Wilk: a small p-value is evidence AGAINST normality
cat("Shapiro-Wilk, raw:      p =", format(shapiro.test(x)$p.value, digits = 3), "\n")
cat("Shapiro-Wilk, log10(x): p =", format(shapiro.test(log10(x))$p.value, digits = 3), "\n")

**How much does a rank-based test cost you?** Less than people fear — about 5% more samples than a
t-test when the data really are normal, and often fewer when they are not. Simulate it rather than
guessing:

In [ ]:
power_sim = function(n, delta, sd, test, reps = 2000) {
  hits = 0
  for (i in 1:reps) {
    a = rnorm(n, 0,     sd)
    b = rnorm(n, delta, sd)
    p = if (test == "t") t.test(a, b, var.equal = TRUE)$p.value else wilcox.test(a, b)$p.value
    hits = hits + (p < 0.05)
  }
  hits / reps
}

for (n in c(10, 15, 20)) {
  cat(sprintf("n = %2d per group:  t-test %.2f   Wilcoxon %.2f\n",
              n, power_sim(n, 1.0, 0.9, "t"), power_sim(n, 1.0, 0.9, "w")))
}

## 7 · When there is no formula: simulate

Formulas exist for the common designs. For anything else — a non-parametric test, a nested design,
a weird outcome distribution, censored values — there is no formula, and you do not need one.

**Power is just "how often would this experiment work?"** So build the experiment in code, run it a
thousand times with the effect you hope to find, and count how often the test comes out significant.
That fraction is the power. It is slower than a formula and correct for every design.

The template below can be bent to any of the six cases: change how the data are made, change the
test, keep the loop.

In [ ]:
simulate_power = function(n, reps = 2000) {
  # EDIT THE TWO MARKED LINES. Everything else stays the same.
  hits = 0
  for (i in 1:reps) {
    # 1. build one fake experiment, with the effect you hope is there -------------
    group_a = rnorm(n, 50, 18)
    group_b = rnorm(n, 60, 18)
    # 2. run exactly the test you plan to run on the real data --------------------
    p = t.test(group_a, group_b, var.equal = TRUE)$p.value
    hits = hits + (p < 0.05)
  }
  hits / reps
}

for (n in c(10, 20, 30, 40, 52)) {
  cat(sprintf("n = %3d per group -> power %.2f\n", n, simulate_power(n)))
}

cat("\nCompare with the formula:",
    round(power.t.test(n = 52, delta = 10, sd = 18, sig.level = 0.05)$power, 2), "at n = 52\n")

---
## Sanity checks before you believe your own number

- **Does n move in the right direction?** Halve the effect and n should roughly quadruple. If not,
  a number is in the wrong place.
- **Is your SD from the right source?** It must describe variation between the *units you are
  counting* — not assay noise, not the spread of a pooled dataset.
- **Have you inflated for dropout, failed runs, contaminated plates?**
- **Is the effect size defensible?** Write the sentence "a difference smaller than this would not
  change what anyone does, because ..." If you cannot finish it, you have not chosen an effect size
  — you have chosen a number.
- **What if you are wrong about it?** Report the n you would need at half the effect. That single
  line is the most honest thing in a power calculation.